# SOTA Ground-Truth Baseline — Optical-DRC LSTM (no conditioning)

A **direct-output** baseline for one-to-one comparison against the
GR-prediction approaches in [`03_initial_GR_pred/`](../03_initial_GR_pred/).

The **model architecture** is a faithful PyTorch port of the **proposed LSTM**
from [Optical-DRC-with-Selective-SSMs](../external/Optical-DRC-with-Selective-SSMs/Code/Models.py)
(`create_model_LSTM`, Simionato et al., JAES 2025) with conditioning removed.
The **training recipe** (loss + optimizer + stateful TBPTT) follows the
[nablafx-for-diffssl-compressor](../external/nablafx-for-diffssl-compressor/)
fork, which trains on this *exact* SSL G-Bus dataset — see note below.

## What is kept identical to `03_initial_GR_pred/train_lstm_nocond.ipynb`

- **Dataset**: Diff-SSL-G-Comp (SSL G-Bus), single setting
  `threshold_-4_attack_1_release_0.4_ratio_10`, no conditioning.
- **Song-level train/val split**: 80/20, `manual_seed(42)` — *exact same
  train/val songs* as the GR notebook.
- **Sample rate**: 44.1 kHz.

## What mirrors the SOTA (and differs from the GR approach)

| | GR predictor (`03_*`) | SOTA LSTM (this) |
|---|---|---|
| **Target** | gain-reduction envelope (dB) | **wet output audio** (direct) |
| **Model** | frame-rate conv+LSTM | sample-rate windowed LSTM (Optical-DRC) |
| **Input framing** | 10 s crop → strided encoder | 64-sample sliding window per step |
| **Training** | random crops, fresh state | **stateful TBPTT**: state carried across consecutive chunks of each track, reset only at track boundary / epoch |
| **Output** | GR_norm, upsampled | `gain × current_sample` (residual) |
| **Loss** | masked L1(dB) + diff-L1 | **masked 0.5·L1 + 0.5·MR-STFT** (nablafx-diffssl) |
| **Metrics** | L1 (dB) | **ESR, RMSE, MSE, MAE** |
| **Optim** | AdamW + ReduceLROnPlateau | **AdamW + ReduceLROnPlateau(0.5, p20)** (nablafx) |

The architecture is the SOTA `create_model_LSTM` minus conditioning:

```
dry [B,1,L]
  → Dense(2) over each 64-sample window      (≡ Conv1d(1,2,kernel=64))
    → LSTM(6)   ── hidden state carried across chunks (stateful) ──┐
      → Dense(2)                                                   │
        → LSTM(6) ── hidden state carried across chunks (stateful) ┘
          → Dense(1)
            → × current input sample          (residual gain → wet)
```

**Two reference recipes — and why we mix them.** The Optical-DRC *paper* trains
this LSTM with plain **MSE**. The nablafx-diffssl fork trains LSTMs on the
*same SSL G-Bus dataset* with **0.5·L1 + 0.5·MR-STFT** + AdamW +
ReduceLROnPlateau. We keep the Optical-DRC **architecture** but adopt the
nablafx **loss + optimizer**, because (a) it is the native recipe for this
dataset, (b) it is not energy-dominated like MSE — so quiet/low-compression
regions are actually fit (the fix for the predicted-GR ceiling), and (c) its
MR-STFT term matches the metric family the GR eval already reports.

**Stateful training** (cells 3 & 5): each batch row is one track; chunks are
fed in time order with the LSTM state propagated chunk→chunk and detached each
step (truncated BPTT), so the model can learn the slow **release recovery**
that fresh random crops could not.

**Runtime**: select **GPU** (*Runtime → Change runtime type*).

In [1]:
# ── 0. Install dependencies ──────────────────────────────────────────
# auraloss provides MultiResolutionSTFTLoss (the nablafx-diffssl freq loss).
!pip install -q lightning torchmetrics soundfile auraloss

In [2]:
# ── 1. Mount Google Drive & locate dataset ───────────────────────────
# Same dataset + setting as 03_initial_GR_pred/train_lstm_nocond.ipynb.

import os
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = None
SETTING = "threshold_-4_attack_1_release_0.4_ratio_10"

def _find_dataset_root() -> str | None:
    import glob as _g
    candidates = _g.glob("/content/drive/*/*/*/Diff-SSL-G-Comp", recursive=False)
    candidates += _g.glob("/content/drive/*/*/Diff-SSL-G-Comp")
    candidates += _g.glob("/content/drive/*/Diff-SSL-G-Comp")
    for c in candidates:
        if (os.path.isdir(os.path.join(c, "processed_normalized"))
                and os.path.isdir(os.path.join(c, "processed_ground_truth"))):
            return c
    return None

if DRIVE_DATA_ROOT is None:
    DRIVE_DATA_ROOT = _find_dataset_root()

DATA_ROOT = DRIVE_DATA_ROOT
assert DATA_ROOT and os.path.isdir(DATA_ROOT), (
    "Dataset folder not found. Hard-set DRIVE_DATA_ROOT above."
)
print(f"Auto-located dataset at: {DATA_ROOT}")

dry_dir = os.path.join(DATA_ROOT, "processed_normalized")
wet_dir = os.path.join(DATA_ROOT, "processed_ground_truth", SETTING)
assert os.path.isdir(dry_dir), f"Missing dry folder: {dry_dir}"
assert os.path.isdir(wet_dir), f"Missing wet folder: {wet_dir}"

import glob
n_dry = len(glob.glob(os.path.join(dry_dir, "*_UnmasteredWAV.wav")))
n_wet = len(glob.glob(os.path.join(wet_dir, "*-exported.wav")))
print(f"Setting    : {SETTING}")
print(f"Dry files  : {n_dry}")
print(f"Wet files  : {n_wet}  (-> usable songs = matched dry/wet pairs)")

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "sota_lstm_runs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs will be saved to: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Auto-located dataset at: /content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp
Setting    : threshold_-4_attack_1_release_0.4_ratio_10
Dry files  : 175
Wet files  : 10  (-> usable songs = matched dry/wet pairs)
Outputs will be saved to: /content/drive/Othercomputers/MacBook Air/data/sota_lstm_runs


In [3]:
# ── 1b. Cache dataset to local SSD ───────────────────────────────────
# Copies the matched dry + wet WAVs from Drive to the Colab SSD for fast
# DataLoader I/O.  (The GR notebook caches .pt GR curves; here we need the
# raw WET audio, since the SOTA model predicts the output directly.)

import shutil, time
from pathlib import Path
from google.colab import drive as _gdrive

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"
USE_LOCAL_CACHE = True

def _remount_drive():
    try:
        _gdrive.flush_and_unmount()
    except Exception:
        pass
    _gdrive.mount("/content/drive", force_remount=True)

def _robust_copy(src: Path, dst: Path, max_retries: int = 5):
    for attempt in range(1, max_retries + 1):
        try:
            with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
                shutil.copyfileobj(fsrc, fdst, length=1024 * 1024)
            return
        except OSError as e:
            print(f"  [retry {attempt}/{max_retries}] {src.name}: {e}")
            try: dst.unlink(missing_ok=True)
            except Exception: pass
            time.sleep(2 * attempt)
            if "Transport endpoint" in str(e) or e.errno in (107, 5):
                _remount_drive()
    raise RuntimeError(f"Failed to copy {src} after {max_retries} retries")

def _mirror(src: Path, dst: Path, label: str, i: int, n: int):
    need = not dst.exists() or dst.stat().st_size != src.stat().st_size
    if need:
        dst.parent.mkdir(parents=True, exist_ok=True)
        _robust_copy(src, dst)
    print(f"  {label}: {i}/{n}  ({src.name}){'' if need else '  [skip]'}")

if USE_LOCAL_CACHE:
    drive_dry = Path(DATA_ROOT) / "processed_normalized"
    drive_wet = Path(DATA_ROOT) / "processed_ground_truth" / SETTING

    # discover matched songs (those with BOTH dry and wet)
    dry_lookup = {p.name.replace("_UnmasteredWAV.wav", ""): p
                  for p in sorted(drive_dry.glob("*_UnmasteredWAV.wav"))}
    songs = []
    for wp in sorted(drive_wet.glob("*-exported.wav")):
        song = wp.name.replace("-exported.wav", "")
        if song in dry_lookup:
            songs.append(song)
    print(f"Found {len(songs)} matched dry/wet songs:")
    for s in songs:
        print(f"  - {s}")

    local_dry = Path(LOCAL_DATA_ROOT) / "processed_normalized"
    local_wet = Path(LOCAL_DATA_ROOT) / "processed_ground_truth" / SETTING

    print(f"\nCopying {len(songs)} dry files ...")
    for i, song in enumerate(songs, 1):
        _mirror(dry_lookup[song], local_dry / dry_lookup[song].name, "dry", i, len(songs))

    print(f"\nCopying {len(songs)} wet files ...")
    for i, song in enumerate(songs, 1):
        src = drive_wet / f"{song}-exported.wav"
        _mirror(src, local_wet / src.name, "wet", i, len(songs))

    DATA_ROOT = LOCAL_DATA_ROOT
    print(f"\nUsing local cache: {DATA_ROOT}")
else:
    print(f"Reading directly from Drive: {DATA_ROOT}")

Found 10 matched dry/wet songs:
  - Air
  - AncoraQui
  - BackroomInTulsa
  - Borderline
  - Ecstasy
  - Electrvm
  - LivingLie
  - NosPalpitants
  - OpenFire
  - SongForJohn

Copying 10 dry files ...
  dry: 1/10  (Air_UnmasteredWAV.wav)  [skip]
  dry: 2/10  (AncoraQui_UnmasteredWAV.wav)  [skip]
  dry: 3/10  (BackroomInTulsa_UnmasteredWAV.wav)  [skip]
  dry: 4/10  (Borderline_UnmasteredWAV.wav)  [skip]
  dry: 5/10  (Ecstasy_UnmasteredWAV.wav)  [skip]
  dry: 6/10  (Electrvm_UnmasteredWAV.wav)  [skip]
  dry: 7/10  (LivingLie_UnmasteredWAV.wav)  [skip]
  dry: 8/10  (NosPalpitants_UnmasteredWAV.wav)  [skip]
  dry: 9/10  (OpenFire_UnmasteredWAV.wav)  [skip]
  dry: 10/10  (SongForJohn_UnmasteredWAV.wav)  [skip]

Copying 10 wet files ...
  wet: 1/10  (Air-exported.wav)  [skip]
  wet: 2/10  (AncoraQui-exported.wav)  [skip]
  wet: 3/10  (BackroomInTulsa-exported.wav)  [skip]
  wet: 4/10  (Borderline-exported.wav)  [skip]
  wet: 5/10  (Ecstasy-exported.wav)  [skip]
  wet: 6/10  (Electrvm-exporte

In [4]:
# ── 2. Imports & SOTA evaluation metrics ─────────────────────────────
# ESR / RMSE replicate external/Optical-DRC-with-Selective-SSMs/Code/Metrics.py
# (PyTorch ports of the Keras-backend metrics).

import torch
import torch.nn as nn
import torch.nn.functional as F

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


def esr_sota(y_true: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
    """Error-to-signal ratio (matches Metrics.ESR)."""
    return torch.mean((y_pred - y_true) ** 2 / (y_true ** 2 + 1e-5))


def rmse_sota(y_true: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
    """Matches Metrics.RMSE: mean(| |pred| - |true| |)."""
    return torch.mean(torch.abs(torch.abs(y_pred) - torch.abs(y_true)))

NVIDIA L4


In [5]:
# ── 3. Stateful dataset & DataModule (SOTA-style sequential streams) ──
#
# Mirrors the Keras stateful data generator in
#   external/Optical-DRC-with-Selective-SSMs/Code/DatasetsClass.py
#
# Instead of random fresh-state crops, we process each track SEQUENTIALLY:
# the LSTM hidden state is carried across consecutive `segment_len` chunks of
# the SAME track (TBPTT), and reset only at track boundaries / epoch start.
# This lets the model learn the slow release-recovery ballistic that fresh
# 186 ms crops could not.
#
# Each row of the batch = one track (B = number of songs in the split), so a
# "step" feeds chunk s of every track in parallel, with state carried step ->
# step.  Shorter tracks are zero-padded and masked out of the loss once they
# end (Diff-SSL songs have unequal lengths; the Keras code assumes equal).
#
# __getitem__(s) returns the WHOLE step-batch:
#   dry   [B, 1, segment_len + window - 1]   (left-padded with window-1 context)
#   wet   [B, 1, segment_len]                (already aligned to the output)
#   mask  [B]                                (1 = real, 0 = past track end)
#   reset (bool)                             (True at s == 0 -> zero LSTM state)

import glob
import soundfile as sf
import torchaudio
import lightning as pl
from torch.utils.data import Dataset, DataLoader
from typing import Optional

SAMPLE_RATE = 44100
WINDOW      = 64       # SOTA input window (samples)
SEGMENT_LEN = 32768     # TBPTT chunk length (SOTA mini_batch_size ~ 2400)


def discover_pairs(data_root: str, settings_folder: str) -> list[dict]:
    dry_dir = os.path.join(data_root, "processed_normalized")
    wet_dir = os.path.join(data_root, "processed_ground_truth", settings_folder)
    dry_lookup = {
        os.path.basename(p).replace("_UnmasteredWAV.wav", ""): p
        for p in sorted(glob.glob(os.path.join(dry_dir, "*_UnmasteredWAV.wav")))
    }
    pairs = []
    for wp in sorted(glob.glob(os.path.join(wet_dir, "*-exported.wav"))):
        song = os.path.basename(wp).replace("-exported.wav", "")
        if song in dry_lookup:
            pairs.append({"song": song, "dry": dry_lookup[song], "wet": wp})
    assert pairs, f"No matched dry/wet pairs in {wet_dir}"
    return sorted(pairs, key=lambda p: p["song"])


class StatefulStepDataset(Dataset):
    """One item == one time-step batch across all tracks (B parallel streams)."""

    def __init__(self, song_meta: list[dict], segment_len: int = SEGMENT_LEN,
                 window: int = WINDOW, sample_rate: int = SAMPLE_RATE):
        self.S = segment_len
        self.w = window
        self.sr = sample_rate

        self.cache: list[dict] = []
        for m in song_meta:
            dry = self._load(m["dry"])
            wet = self._load(m["wet"])
            n = min(dry.shape[-1], wet.shape[-1])
            self.cache.append({
                "song": m["song"], "dry": dry[..., :n].contiguous(),
                "wet": wet[..., :n].contiguous(), "K": n // self.S,
            })

        self.B = len(self.cache)
        self.K = max(c["K"] for c in self.cache)
        ram = sum(c["dry"].numel() + c["wet"].numel() for c in self.cache) * 4 / 1024**2
        print(f"StatefulStepDataset: B={self.B} streams, {self.K} steps/epoch, "
              f"{ram:.0f} MB cached  [segment_len={self.S}, window={self.w}]")
        for c in self.cache:
            print(f"    {c['song']:<20} {c['K']:>5} chunks ({c['dry'].shape[-1]/self.sr:6.1f}s)")

    def _load(self, path: str) -> torch.Tensor:
        a, sr = sf.read(path, dtype="float32", always_2d=True)
        a = torch.from_numpy(a.T)
        if sr != self.sr:
            a = torchaudio.functional.resample(a, sr, self.sr)
        if a.shape[0] > 1:
            a = a.mean(dim=0, keepdim=True)
        return a

    def __len__(self):
        return self.K

    def __getitem__(self, s: int):
        S, w, B = self.S, self.w, self.B
        dry_b = torch.zeros(B, 1, S + w - 1)
        wet_b = torch.zeros(B, 1, S)
        mask = torch.zeros(B)
        for r, c in enumerate(self.cache):
            if s < c["K"]:
                o = s * S
                wet_b[r] = c["wet"][:, o:o + S]
                lo = o - (w - 1)
                if lo < 0:                       # first chunk: zero-pad left context
                    seg = c["dry"][:, :o + S]
                    dry_b[r, :, (w - 1) - o:] = seg
                else:
                    dry_b[r] = c["dry"][:, lo:o + S]
                mask[r] = 1.0
        return dry_b, wet_b, mask, (s == 0)


class StatefulSOTADataModule(pl.LightningDataModule):
    def __init__(self, data_root, settings_folder, segment_len=SEGMENT_LEN,
                 window=WINDOW, sample_rate=SAMPLE_RATE, train_split=0.8):
        super().__init__()
        self.save_hyperparameters()
        self.data_root = data_root
        self.settings_folder = settings_folder
        self.segment_len = segment_len
        self.window = window
        self.sample_rate = sample_rate
        self.train_split = train_split

    def setup(self, stage: Optional[str] = None) -> None:
        pairs = discover_pairs(self.data_root, self.settings_folder)
        songs = sorted(p["song"] for p in pairs)
        if len(songs) < 2:
            raise ValueError("Need >= 2 songs for train/val split.")
        # IDENTICAL split logic to 03_initial_GR_pred (seed 42)
        generator = torch.Generator().manual_seed(42)
        perm = torch.randperm(len(songs), generator=generator).tolist()
        n_train = int(len(songs) * self.train_split)
        n_train = min(max(1, n_train), len(songs) - 1)
        train_songs = {songs[i] for i in perm[:n_train]}
        val_songs = set(songs) - train_songs

        train_meta = [p for p in pairs if p["song"] in train_songs]
        val_meta = [p for p in pairs if p["song"] in val_songs]
        print(f"Train songs ({len(train_meta)}): {sorted(train_songs)}")
        self.train_dataset = StatefulStepDataset(train_meta, self.segment_len, self.window, self.sample_rate)
        print(f"Val songs ({len(val_meta)}): {sorted(val_songs)}")
        self.val_dataset = StatefulStepDataset(val_meta, self.segment_len, self.window, self.sample_rate)

    # batch_size=None: the dataset already returns whole step-batches.
    # shuffle MUST be False to preserve time-order for state continuity.
    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=None, shuffle=False,
                          num_workers=0, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=None, shuffle=False,
                          num_workers=0, pin_memory=True)

In [6]:
# ── 4. SOTA LSTM model (PyTorch port, conditioning removed) ──────────
#
# Faithful port of create_model_LSTM from
#   external/Optical-DRC-with-Selective-SSMs/Code/Models.py
# with FiLM / TemporalFiLM / FFT-feature conditioning stripped out.
#
#   Keras                                  ->  here
#   Dense(2) over [.,.,64] window          ->  Conv1d(1, 2, kernel=64, stride=1)
#   LSTM(6, stateful, return_sequences)    ->  nn.LSTM(2, 6)   (state carried)
#   Dense(2)                               ->  nn.Linear(6, 2)
#   FiLM(z1) -> TemporalFiLM(z2)           ->  (removed: no conditioning)
#   LSTM(6, stateful, ...)                 ->  nn.LSTM(2, 6)   (state carried)
#   Dense(1)                               ->  nn.Linear(6, 1)
#   Multiply([inp[:, -1], x])              ->  current_sample * gain   (residual)
#
# The forward optionally accepts/returns LSTM state so the training loop can
# carry it across consecutive chunks of a track (SOTA stateful processing).
# Called without state (state=None) it behaves statelessly — used by eval.


class OpticalLSTM_NoCond(nn.Module):
    def __init__(self, window: int = WINDOW, units: int = 6):
        super().__init__()
        self.window = window
        self.proj = nn.Conv1d(1, 2, kernel_size=window, stride=1)  # Dense(2) over window
        self.lstm1 = nn.LSTM(2, units, batch_first=True)
        self.dense_mid = nn.Linear(units, 2)
        self.lstm2 = nn.LSTM(2, units, batch_first=True)
        self.dense_out = nn.Linear(units, 1)

    def forward(self, dry: torch.Tensor, state=None, return_state: bool = False):
        # dry: [B, 1, L]  with L = segment_len + window - 1
        cur = dry[:, :, self.window - 1:]   # [B, 1, S]  current (most-recent) sample
        x = self.proj(dry).transpose(1, 2)  # [B, S, 2]   (Dense over each 64-window)
        s1, s2 = (None, None) if state is None else state
        x, s1 = self.lstm1(x, s1)           # [B, S, units]   (h1, c1 carried)
        x = self.dense_mid(x)               # [B, S, 2]
        x, s2 = self.lstm2(x, s2)           # [B, S, units]   (h2, c2 carried)
        x = self.dense_out(x).transpose(1, 2)   # [B, 1, S]
        out = cur * x                       # [B, 1, S]  residual gain -> wet
        return (out, (s1, s2)) if return_state else out

In [7]:
# ── 5. Lightning System (stateful TBPTT, nablafx-diffssl loss) ───────
#
# Loss: nablafx-for-diffssl's TimeAndFrequencyDomainLoss
#       = 0.5 * L1(time) + 0.5 * MultiResolutionSTFTLoss(freq)
#   (external/nablafx-for-diffssl-compressor/nablafx/loss.py + LSTM configs)
#   — chosen over the Optical-DRC MSE because it is the native recipe for THIS
#   dataset, is not energy-dominated (so quiet/low-GR regions are fit), and
#   matches the MR-STFT family used by the GR eval metrics.
# Masked over zero-padded rows (rows whose track has ended).
#
# Optimizer: AdamW(lr, betas=(0.9,0.999), eps=1e-8)  +  ReduceLROnPlateau(
#   mode="min", factor=0.5, patience=20) on loss/val  — exactly nablafx.
#
# State: SOTA-style stateful TBPTT — LSTM state carried across consecutive
# chunks of a track, detached each step, reset at track boundary / epoch.

import auraloss
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor


class SOTALSTMSystem(pl.LightningModule):
    def __init__(self, model: nn.Module, window: int = WINDOW, lr: float = 1e-3):
        super().__init__()
        self.model = model
        self.window = window
        self.lr = lr
        # nablafx-diffssl TimeAndFrequencyDomainLoss (0.5 / 0.5)
        self.l1 = nn.L1Loss()
        self.mrstft = auraloss.freq.MultiResolutionSTFTLoss(
            fft_sizes=[1024, 2048, 512],
            hop_sizes=[120, 240, 50],
            win_lengths=[600, 1200, 240],
            w_sc=1.0, w_log_mag=1.0, w_lin_mag=0.0,
        )
        self.td_w, self.fd_w = 0.5, 0.5
        self._train_state = None
        self._val_state = None

    def forward(self, x):           # stateless convenience (used by eval notebook)
        return self.model(x)

    @staticmethod
    def _detach(state):
        if state is None:
            return None
        d = lambda s: None if s is None else (s[0].detach(), s[1].detach())
        return (d(state[0]), d(state[1]))

    def _step(self, dry, wet, mask, state):
        pred, new_state = self.model(dry, state, return_state=True)   # [B,1,S]
        valid = mask.bool()
        pv, tv = pred[valid], wet[valid]            # drop zero-padded (ended) rows

        td = self.l1(pv, tv)
        fd = self.mrstft(pv, tv)
        loss = self.td_w * td + self.fd_w * fd

        with torch.no_grad():
            mae, esr, rmse = F.l1_loss(pv, tv), esr_sota(tv, pv), rmse_sota(tv, pv)
        return loss, td, fd, mae, esr, rmse, new_state

    def _log_all(self, mode, loss, td, fd, mae, esr, rmse, bs):
        self.log(f"loss/{mode}", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=bs)
        self.log(f"loss/{mode}_td", td, on_step=False, on_epoch=True, batch_size=bs)
        self.log(f"loss/{mode}_fd", fd, on_step=False, on_epoch=True, batch_size=bs)
        self.log(f"mae/{mode}", mae, on_step=False, on_epoch=True, batch_size=bs)
        self.log(f"esr/{mode}", esr, on_step=False, on_epoch=True,
                 prog_bar=(mode == "val"), batch_size=bs)
        self.log(f"rmse/{mode}", rmse, on_step=False, on_epoch=True, batch_size=bs)

    def training_step(self, batch, batch_idx):
        dry, wet, mask, reset = batch
        if bool(reset):
            self._train_state = None
        loss, td, fd, mae, esr, rmse, st = self._step(dry, wet, mask, self._train_state)
        self._train_state = self._detach(st)
        self._log_all("train", loss, td, fd, mae, esr, rmse, int(mask.sum().item()))
        return loss

    def validation_step(self, batch, batch_idx):
        dry, wet, mask, reset = batch
        if bool(reset):
            self._val_state = None
        loss, td, fd, mae, esr, rmse, st = self._step(dry, wet, mask, self._val_state)
        self._val_state = self._detach(st)
        self._log_all("val", loss, td, fd, mae, esr, rmse, int(mask.sum().item()))
        return loss

    def on_train_epoch_start(self):
        self._train_state = None

    def on_validation_epoch_start(self):
        self._val_state = None

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, betas=(0.9, 0.999), eps=1e-8)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=20)
        return {"optimizer": opt,
                "lr_scheduler": {"scheduler": sched, "monitor": "loss/val",
                                 "interval": "epoch", "frequency": 1}}

In [9]:
# ── 6. Train ─────────────────────────────────────────────────────────

import time, json
from datetime import datetime
from lightning.pytorch.callbacks import EarlyStopping, TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

RESUME_RUN: str | None = "sota_lstm_20260605_112404_sota_lstm_stateful_l1mrstft_longer_segment"
RUN_TAG: str = "sota_lstm_stateful_l1mrstft_longer_segment"

# ── hparams ──────────────────────────────────────────────────────────
LR                  = 1e-3      # nablafx-diffssl LSTM default
MAX_EPOCHS          = 1000
EARLY_STOP_PATIENCE = 50
UNITS               = 6         # SOTA (Optical-DRC) LSTM units
CKPT_EVERY_N_EPOCHS = 5
# Batch size is implicit: one parallel stream per training song (B = #songs).
# ─────────────────────────────────────────────────────────────────────

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    assert os.path.isfile(_resume_ckpt), f"No last.ckpt in {RUN_DIR}"
    print(f"RESUMING run: {RUN_NAME}")
else:
    _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_NAME = f"sota_lstm_{_ts}_{RUN_TAG}" if RUN_TAG else f"sota_lstm_{_ts}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
assert "/drive/" not in DATA_ROOT, f"DATA_ROOT still on Drive ({DATA_ROOT}). Run cell 1b."

dm = StatefulSOTADataModule(
    data_root=DATA_ROOT, settings_folder=SETTING,
    segment_len=SEGMENT_LEN, window=WINDOW, sample_rate=SAMPLE_RATE, train_split=0.8,
)
dm.setup()

B_train = dm.train_dataset.B
steps_per_epoch = len(dm.train_dataset)        # = K chunks (state carried across them)

model = OpticalLSTM_NoCond(window=WINDOW, units=UNITS)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model units      : {UNITS}")
print(f"Parameters       : {n_params:,}")
print(f"Parallel streams : {B_train} (one per training song)")
print(f"Steps/epoch      : {steps_per_epoch}")

# ── save hparams ─────────────────────────────────────────────────────
_hparams = {
    "approach": "sota_direct_output_lstm",
    "model_type": "sota_direct_output_lstm",
    "source_model": "Optical-DRC create_model_LSTM (no conditioning)",
    "training": "stateful_tbptt_per_track",       # state carried across chunks
    "sample_rate": SAMPLE_RATE, "window": WINDOW, "segment_len": SEGMENT_LEN,
    "setting": SETTING,
    "split_unit": "song", "split_seed": 42, "train_split": 0.8,
    "parallel_streams": B_train, "lr": LR, "units": UNITS,
    "state_reset": "track_boundary_and_epoch",
    "max_epochs": MAX_EPOCHS, "early_stop_patience": EARLY_STOP_PATIENCE,
    "loss": "nablafx_0.5*L1 + 0.5*MR-STFT",
    "loss_ref": "nablafx-for-diffssl TimeAndFrequencyDomainLoss",
    "metrics": ["mse", "mae", "esr", "rmse"],
    "optimizer": "adamw + reducelronplateau(0.5,p20) [nablafx]",
    "num_params": n_params,
}
with open(os.path.join(RUN_DIR, "hparams.json"), "w") as _f:
    json.dump(_hparams, _f, indent=2)
print("Saved hparams.json")

system = SOTALSTMSystem(model=model, window=WINDOW, lr=LR)

# ── callbacks & loggers ──────────────────────────────────────────────
ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
best_cb = ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min",
                          save_top_k=3, save_last=True,
                          filename="best-{epoch:03d}-{step}", auto_insert_metric_name=False)
periodic_cb = ModelCheckpoint(dirpath=ckpt_dir, every_n_epochs=CKPT_EVERY_N_EPOCHS,
                              save_top_k=-1, filename="epoch-{epoch:03d}",
                              auto_insert_metric_name=False)
lr_cb = LearningRateMonitor(logging_interval="epoch")
early_stop_cb = EarlyStopping(monitor="loss/val", mode="min",
                              patience=EARLY_STOP_PATIENCE, min_delta=0.0, verbose=False)
tb_logger = TensorBoardLogger(save_dir=RUN_DIR, name="tb", version="")
csv_logger = CSVLogger(save_dir=RUN_DIR, name="csv", version="")

RESUMING run: sota_lstm_20260605_112404_sota_lstm_stateful_l1mrstft_longer_segment
Train songs (8): ['Air', 'AncoraQui', 'BackroomInTulsa', 'Ecstasy', 'Electrvm', 'LivingLie', 'OpenFire', 'SongForJohn']
StatefulStepDataset: B=8 streams, 476 steps/epoch, 731 MB cached  [segment_len=32768, window=64]
    Air                    263 chunks ( 195.5s)
    AncoraQui              314 chunks ( 233.5s)
    BackroomInTulsa        370 chunks ( 274.9s)
    Ecstasy                346 chunks ( 257.7s)
    Electrvm               347 chunks ( 258.3s)
    LivingLie              408 chunks ( 303.3s)
    OpenFire               476 chunks ( 354.0s)
    SongForJohn            396 chunks ( 294.7s)
Val songs (2): ['Borderline', 'NosPalpitants']
StatefulStepDataset: B=2 streams, 368 steps/epoch, 159 MB cached  [segment_len=32768, window=64]
    Borderline             368 chunks ( 273.9s)
    NosPalpitants          266 chunks ( 198.1s)
Model units      : 6
Parameters       : 631
Parallel streams : 8 (one per tr

In [ ]:
# ── 7. Launch TensorBoard & fit ──────────────────────────────────────

_tb_link = "/content/tb_current"
if os.path.islink(_tb_link) or os.path.exists(_tb_link):
    os.remove(_tb_link)
os.symlink(os.path.join(RUN_DIR, "tb"), _tb_link)

%load_ext tensorboard
%tensorboard --logdir /content/tb_current

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto", devices="auto",
    precision="32-true",               # fp32: stable MR-STFT, model is tiny (~631 params)
    callbacks=[best_cb, periodic_cb, lr_cb, early_stop_cb,
               TQDMProgressBar(refresh_rate=10)],
    logger=[tb_logger, csv_logger],
    log_every_n_steps=10,
    default_root_dir=RUN_DIR,
    gradient_clip_val=1.0,             # SOTA Adam(clipnorm=1)
    gradient_clip_algorithm="norm",
    use_distributed_sampler=False,     # keep time-order intact for stateful streams
)

t0 = time.time()
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"\nTotal training time: {(time.time() - t0)/60:.1f} min")
print(f"Best val loss (L1+MR-STFT): {best_cb.best_model_score:.6f}")
print(f"Best ckpt    : {best_cb.best_model_path}")

: 

In [ ]:
# ── 8. Evaluate: predicted vs target output waveform + metrics ───────

import matplotlib.pyplot as plt
import numpy as np

ckpt = torch.load(best_cb.best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(ckpt["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {best_cb.best_model_path}")

# Stateful val loader now yields (dry, wet, mask, reset) step-batches.
# wet is already aligned to the output (length = segment_len); dry carries the
# extra window-1 left context.  Pick a mid-track step so state has settled.
val_steps = list(dm.val_dataloader())
batch = val_steps[min(len(val_steps) // 2, len(val_steps) - 1)]
dry, wet, mask = batch[0].cuda(), batch[1].cuda(), batch[2].cuda()

with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
    pred = system(dry)                          # [B,1,S]
target = wet                                    # already aligned, [B,1,S]
n = min(pred.shape[-1], target.shape[-1])
pred, target = pred[..., :n], target[..., :n]

valid = mask.bool()
pv, tv = pred[valid], target[valid]
print(f"Val step  MSE={F.mse_loss(pv, tv):.6f}  MAE={F.l1_loss(pv, tv):.6f}  "
      f"ESR={esr_sota(tv, pv):.6f}  RMSE={rmse_sota(tv, pv):.6f}")

pred_np = pred.float().cpu().numpy()
tgt_np = target.float().cpu().numpy()
dry_np = dry[:, :, WINDOW - 1:WINDOW - 1 + n].float().cpu().numpy()

rows = torch.nonzero(valid).squeeze(1).tolist()
n_plots = min(4, len(rows))
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots), sharex=True)
if n_plots == 1:
    axes = [axes]
t = np.arange(n) / SAMPLE_RATE
for ax, i in zip(axes, rows[:n_plots]):
    ax.plot(t, dry_np[i, 0], label="Dry (in)", alpha=0.35, linewidth=0.4, color="gray")
    ax.plot(t, tgt_np[i, 0], label="Target (wet)", alpha=0.8, linewidth=0.5)
    ax.plot(t, pred_np[i, 0], label="Predicted", alpha=0.8, linewidth=0.5)
    e = float(np.mean(np.abs(pred_np[i, 0] - tgt_np[i, 0])))
    ax.set_ylabel("amp"); ax.set_title(f"Stream {i} — MAE = {e:.4f}")
    ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)
axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"SOTA LSTM (stateful) direct-output — best val MSE {best_cb.best_model_score:.6f}", y=1.01)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()